# Release notebook

This output-free copy corresponds to the final reported protocol. Exploratory results are not part of this repository. Use the frozen settings and artifact hashes in `configs/final_experiment.json`.


# 2단계: 독립 9,000건 / Llama 3 Direct · CoT · 최종 SELF-DISCOVER v6c

**새 GPU 런타임에서 위부터 모두 실행. Drive는 제한 재현 자료에 접근 권한이 있는 계정으로 연결.**
완료된 1단계 출력과 handoff를 자동으로 읽는다. 파일 업로드와 경로 수정은 필요 없다.

- 최종 DistilBERT가 라우팅한 동일 사례만 세 방식으로 재평가한다. 전체 성능의 분모는 9,000건이다.
- Direct/CoT는 기존 최종 프롬프트, v6c는 동료에게 받은 최종 공통 계획과 프롬프트를 그대로 사용한다. 공통 계획을 새로 생성하지 않는다.
- 원문 입력, 4-bit NF4/fp16, greedy decoding. Direct/CoT 각 호출 상한 1,024, v6c 1,536토큰으로 기존 설정을 유지한다.
- 파싱 실패는 Phase 1 라벨을 유지한다. 자동 재생성이나 정답을 보고 라벨을 고치는 단계는 없다.
- 생성 호출별·사례별 결과를 저장한다. 중단되면 같은 GPU 종류와 환경에서 모두 실행하면 완료된 호출과 사례를 건너뛴다.
- 이 노트북은 분류 모델 학습·추론·보정·임계값 탐색을 다시 하지 않는다.


## 1. 설치
새 런타임에서도 반드시 실행한다. HF_TOKEN은 선택 사항이며, 없다고 Drive 오류로 처리하지 않는다.

In [ ]:
%pip install -q transformers==5.17.0 bitsandbytes==0.50.2 accelerate==1.14.0 sentencepiece protobuf pandas==2.2.3 numpy==2.1.3 scipy==1.16.3 scikit-learn==1.6.1 tqdm


## 2. Drive 및 고정 프로토콜

In [ ]:
import gc, hashlib, importlib.metadata, json, os, re, time
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
import torch
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from google.colab import drive, userdata
drive.mount("/content/drive")
try:
    token = userdata.get("HF_TOKEN")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    token = None
if token:
    os.environ["HF_TOKEN"] = token
BASE = Path("/content/drive/MyDrive/confidence_guided_llm_reasoning/outputs_final/unified_final/additional_9000_20260930")
PHASE1_DIR = BASE / "phase1"
OUTPUT_DIR = BASE / "phase2/llama3_direct_cot_v6c_v1"
LABELS = ["Depression", "Neutral", "Happy"]
BASE_SEED = 42
MAX_ROWS = None
MAX_REASONING_CHARACTERS = 6000
REASONING_HEAD_CHARACTERS = 3500
REASONING_TAIL_CHARACTERS = 2500
LOAD_IN_4BIT = True
LLAMA3_MAX_NEW_TOKENS = 1024
V6_MAX_NEW_TOKENS = 1536
MODEL_NAME = "NousResearch/Meta-Llama-3-8B-Instruct"
MODEL_REVISION = "53346005fb0ef11d3b6a83b12c895cca40156b6c"


In [ ]:
def normalize_bool(value):
    if isinstance(value, bool):
        return value
    text = str(value).strip().lower()
    if text in {"true", "1", "yes", "y"}:
        return True
    if text in {"false", "0", "no", "n"}:
        return False
    raise ValueError(f"Unrecognized Boolean value: {value!r}")

def normalize_label(value):
    text = str(value).strip().lower()
    for label in LABELS:
        if text == label.lower():
            return label
    return np.nan

def limit_reasoning_text(value):
    value = "" if pd.isna(value) else str(value)
    if len(value) <= MAX_REASONING_CHARACTERS:
        return value, False
    marker = "\n\n[Middle omitted to fit model context; beginning and ending preserved.]\n\n"
    available = MAX_REASONING_CHARACTERS - len(marker)
    head_characters = min(REASONING_HEAD_CHARACTERS, available)
    tail_characters = min(REASONING_TAIL_CHARACTERS, available - head_characters)
    return (
        value[:head_characters]
        + marker
        + value[-tail_characters:],
        True,
    )

def stable_seed(example_id, method, stage):
    key = f"{BASE_SEED}:{example_id}:{method}:{stage}".encode("utf-8")
    return int.from_bytes(hashlib.sha256(key).digest()[:4], "big")

def set_generation_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def load_chat_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(
        model_name, revision=MODEL_REVISION, trust_remote_code=True, token=os.environ.get("HF_TOKEN")
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    quantization_config = None
    if LOAD_IN_4BIT:
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        revision=MODEL_REVISION,
        torch_dtype=torch.float16,
        device_map="auto",
        quantization_config=quantization_config,
        trust_remote_code=True,
        token=os.environ.get("HF_TOKEN"),
    )
    model.eval()
    return tokenizer, model

def format_messages_without_chat_template(messages):
    system_parts = [m["content"] for m in messages if m.get("role") == "system"]
    dialogue = [m for m in messages if m.get("role") != "system"]
    system_text = "\n".join(system_parts).strip()
    prompt, pending_user, first_turn = "", None, True
    for message in dialogue:
        role = message.get("role")
        content = str(message.get("content", "")).strip()
        if role == "user":
            pending_user = content
        elif role == "assistant" and pending_user is not None:
            if first_turn and system_text:
                prompt += f"<s>[INST] <<SYS>>\n{system_text}\n<</SYS>>\n\n{pending_user} [/INST] {content} </s>"
            else:
                prompt += f"<s>[INST] {pending_user} [/INST] {content} </s>"
            pending_user, first_turn = None, False
    if pending_user is not None:
        if first_turn and system_text:
            prompt += f"<s>[INST] <<SYS>>\n{system_text}\n<</SYS>>\n\n{pending_user} [/INST]"
        else:
            prompt += f"<s>[INST] {pending_user} [/INST]"
    return prompt

def build_chat_inputs(tokenizer, model, messages):
    if getattr(tokenizer, "chat_template", None):
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    else:
        prompt = format_messages_without_chat_template(messages)
    encoded = tokenizer(prompt, return_tensors="pt")
    return {key: value.to(model.device) for key, value in encoded.items()}

def chat_generate(
    tokenizer, model, messages, *, max_new_tokens, seed,
    do_sample=False, temperature=None, top_p=None,
):
    set_generation_seed(seed)
    model_inputs = build_chat_inputs(tokenizer, model, messages)
    input_ids = model_inputs["input_ids"]
    context = int(model.config.max_position_embeddings)
    if input_ids.shape[-1] + max_new_tokens > context:
        raise ValueError(f"Context exceeded: {input_ids.shape[-1]} + {max_new_tokens} > {context}. No silent truncation.")
    terminators = [tokenizer.eos_token_id]
    if "<|eot_id|>" in tokenizer.get_vocab():
        eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")
        if isinstance(eot_id, int) and eot_id >= 0 and eot_id not in terminators:
            terminators.append(eot_id)
    kwargs = dict(
        **model_inputs,
        max_new_tokens=max_new_tokens,
        eos_token_id=terminators,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=do_sample,
    )
    if do_sample and temperature is not None:
        kwargs["temperature"] = temperature
    if do_sample and top_p is not None:
        kwargs["top_p"] = top_p
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    started = time.perf_counter()
    with torch.inference_mode():
        outputs = model.generate(**kwargs)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    generated = outputs[0][input_ids.shape[-1]:]
    return {
        "text": tokenizer.decode(generated, skip_special_tokens=True).strip(),
        "input_tokens": int(input_ids.shape[-1]),
        "generated_tokens": int(generated.shape[-1]),
        "elapsed_seconds": float(elapsed),
        "hit_token_limit": bool(generated.shape[-1] >= max_new_tokens),
    }

def parse_final_label(output):
    text = str(output)
    match = re.search(
        r"Final\s+label\s*\*{0,2}\s*:\s*\*{0,2}\s*"
        r"(Depression|Neutral|Happy)\b",
        text,
        flags=re.I,
    )
    if match:
        return normalize_label(match.group(1))

    # Some models return a clear terminal label without the requested prefix.
    # Recover it only when the final-stage answer contains one unique allowed
    # label; refusals and outputs mentioning multiple labels remain unparsed.
    labels = {
        normalize_label(label)
        for label in re.findall(r"\b(Depression|Neutral|Happy)\b", text, flags=re.I)
    }
    return next(iter(labels)) if len(labels) == 1 else np.nan

def summarize_method(full_phase1, routed_results, prediction_col, method):
    processed_ids = set(routed_results["example_id"].astype(str))
    evaluation_base = full_phase1
    scope = "full_end_to_end"
    if MAX_ROWS is not None:
        evaluation_base = full_phase1[full_phase1["example_id"].isin(processed_ids)].copy()
        scope = "smoke_routed_subset"
    merged = evaluation_base.merge(
        routed_results[["example_id", prediction_col]],
        on="example_id", how="left", validate="one_to_one",
    )
    merged["final_label"] = merged["phase1_label"]
    routed_mask = merged["phase1_routed"]
    parsed_routed_mask = routed_mask & merged[prediction_col].isin(LABELS)
    merged.loc[parsed_routed_mask, "final_label"] = merged.loc[
        parsed_routed_mask, prediction_col
    ]
    merged["phase2_parse_success"] = ~routed_mask | parsed_routed_mask
    merged["phase2_fallback_to_phase1"] = routed_mask & ~parsed_routed_mask
    merged["phase1_correct"] = merged["phase1_label"].eq(merged["target_label"])
    merged["final_correct"] = merged["final_label"].eq(merged["target_label"])
    corrected = int((routed_mask & ~merged["phase1_correct"] & merged["final_correct"]).sum())
    introduced = int((routed_mask & merged["phase1_correct"] & ~merged["final_correct"]).sum())
    parsed = routed_results[prediction_col].isin(LABELS)
    summary = {
        "method": method,
        "scope": scope,
        "total_rows": len(merged),
        "routed_rows": int(routed_mask.sum()),
        "parsed_routed_rows": int(parsed.sum()),
        "parse_failures": int((~parsed).sum()),
        "phase1_accuracy": float(merged["phase1_correct"].mean()),
        "end_to_end_accuracy": float(merged["final_correct"].mean()),
        "end_to_end_macro_f1": float(
            f1_score(merged["target_label"], merged["final_label"], labels=LABELS, average="macro")
        ),
        "corrected": corrected,
        "introduced": introduced,
        "net_corrections": corrected - introduced,
    }
    return merged, summary

CLASSIFICATION_POLICY = """Classification Policy:
- Depression: unresolved sadness, hopelessness, emotional distress, emotional exhaustion, withdrawal, self-devaluation, or a clearly negative overall trajectory is dominant.
- Neutral: the text is mainly factual, routine, balanced, informational, or emotionally mild, without a dominant positive or distress-related state.
- Happy: happiness, relief, gratitude, accomplishment, fulfillment, or a clearly positive resolution is dominant.

Assess the dominant emotional meaning of the full text. For mixed or shifting emotions, consider the overall trajectory and final takeaway. Do not decide from isolated words, and do not make clinical diagnoses or treatment recommendations."""

DIRECT_PROMPT_VERSION = "final-llama3-direct-v1-original-text"

DIRECT_TEMPLATE = """You are an expert annotator for research-oriented, non-clinical emotion classification.

{policy}

Text:
{text}

The Phase 1 classifier predicted: {phase1_label}

Select the final label that best represents the text. Return no rationale and end with exactly one line in this form:
Final label: Depression
or Final label: Neutral
or Final label: Happy"""

COT_PROMPT_VERSION = "final-llama3-cot-v2-original-text"

COT_REQUESTS = [
    "You are an expert annotator for research-oriented, non-clinical emotion classification. Assist in analyzing emotions in text data. Do not make a clinical diagnosis, infer a medical condition, or provide treatment advice.",
    "I will first provide a piece of text. Independently assess its emotional content before comparing it with a Phase 1 AI-generated label. The only permitted labels are Depression, Neutral, and Happy.",
    "Independently analyze the text. State the dominant emotion and textual evidence before considering the Phase 1 label.\n\n{policy}\n\nUse only Depression, Neutral, or Happy for your provisional label.",
    "The Phase 1 classifier predicted: {phase1_label}\n\nCompare that prediction with your independent assessment. Confirm it only when it is supported by the dominant emotional meaning of the full text. Otherwise, explain the correction using textual evidence only.",
    "Provide the final Phase 2 decision. Use only one exact label: Depression, Neutral, or Happy. Do not use synonyms or additional labels. End the response with exactly one line: Final label: [label]",
]

def run_llama3_direct(tokenizer, model, row):
    prompt = DIRECT_TEMPLATE.format(
        policy=CLASSIFICATION_POLICY,
        text=row["phase2_original_text"],
        phase1_label=row["phase1_label"],
    )
    result = chat_generate(
        tokenizer, model, [{"role": "user", "content": prompt}],
        max_new_tokens=LLAMA3_MAX_NEW_TOKENS,
        seed=stable_seed(row["example_id"], "llama3_direct", "final"),
        do_sample=False,
    )
    return {
        "direct_answer": result["text"],
        "final_label": parse_final_label(result["text"]),
        "input_tokens_total": result["input_tokens"],
        "generated_tokens_total": result["generated_tokens"],
        "generation_seconds_total": result["elapsed_seconds"],
        "any_stage_hit_token_limit": result["hit_token_limit"],
    }

def run_llama3_cot(tokenizer, model, row):
    example_id = row["example_id"]
    messages = [
        {"role": "system", "content": COT_REQUESTS[0]},
        {"role": "user", "content": COT_REQUESTS[1]},
    ]
    stages = []
    ack = chat_generate(
        tokenizer, model, messages, max_new_tokens=LLAMA3_MAX_NEW_TOKENS,
        seed=stable_seed(example_id, "llama3_cot", "ack"), do_sample=False,
    )
    stages.append(ack)
    messages += [
        {"role": "assistant", "content": ack["text"]},
        {"role": "user", "content": f"Text:\n{row['phase2_original_text']}"},
    ]
    text_response = chat_generate(
        tokenizer, model, messages, max_new_tokens=LLAMA3_MAX_NEW_TOKENS,
        seed=stable_seed(example_id, "llama3_cot", "text_response"), do_sample=False,
    )
    stages.append(text_response)
    messages += [
        {"role": "assistant", "content": text_response["text"]},
        {"role": "user", "content": COT_REQUESTS[2].format(policy=CLASSIFICATION_POLICY)},
    ]
    independent = chat_generate(
        tokenizer, model, messages, max_new_tokens=LLAMA3_MAX_NEW_TOKENS,
        seed=stable_seed(example_id, "llama3_cot", "independent"), do_sample=False,
    )
    stages.append(independent)
    messages += [
        {"role": "assistant", "content": independent["text"]},
        {"role": "user", "content": COT_REQUESTS[3].format(phase1_label=row["phase1_label"])},
    ]
    comparison = chat_generate(
        tokenizer, model, messages, max_new_tokens=LLAMA3_MAX_NEW_TOKENS,
        seed=stable_seed(example_id, "llama3_cot", "comparison"), do_sample=False,
    )
    stages.append(comparison)
    messages += [
        {"role": "assistant", "content": comparison["text"]},
        {"role": "user", "content": COT_REQUESTS[4]},
    ]
    final = chat_generate(
        tokenizer, model, messages, max_new_tokens=LLAMA3_MAX_NEW_TOKENS,
        seed=stable_seed(example_id, "llama3_cot", "final"), do_sample=False,
    )
    stages.append(final)
    return {
        "cot_ack": ack["text"],
        "cot_text_response": text_response["text"],
        "cot_independent": independent["text"],
        "cot_comparison": comparison["text"],
        "cot_final_answer": final["text"],
        "final_label": parse_final_label(final["text"]),
        "input_tokens_total": sum(item["input_tokens"] for item in stages),
        "generated_tokens_total": sum(item["generated_tokens"] for item in stages),
        "generation_seconds_total": sum(item["elapsed_seconds"] for item in stages),
        "any_stage_hit_token_limit": any(item["hit_token_limit"] for item in stages),
    }
ANNOTATOR_SYSTEM_PROMPT = (
    "You are an expert annotator for research-oriented, non-clinical emotion classification. "
    "Do not provide clinical diagnosis, medical inference, treatment advice, or professional "
    "mental health advice."
)

V6_TEMPLATE = """Annotate one social-media post for a research-oriented, non-clinical emotion study.

{policy}

Reasoning structure to follow:
{structure}

Post:
<post>
{text}
</post>

Work through the reasoning structure on this post. For every step, quote the exact words from the
post that your finding rests on. Do not decide the label before the final step.

Time frame and whose state it is. Before weighing anything, establish two things and quote the
evidence for each:
- Is the emotional state the writer reports about their present, or about a past they are describing
  from a changed position?
- Is it the writer's own state, or a state they attribute to someone else, to a quotation, or to a
  hypothetical?
These cut in every direction. A post can describe severe hardship and still be Happy when the writer
presents it as past and resolved. A post can be light or matter-of-fact in tone and still be
Depression when the distress is present and unresolved. A post can recount someone else's crisis
while the writer is composed, which is Neutral.

Evidence ledger. Search just as hard for each of the three labels; do not give a shorter treatment to
the ones you find less likely.

Depression - strongest supporting quote: "..." | strongest quote against: "..."
Neutral - strongest supporting quote: "..." | strongest quote against: "..."
Happy - strongest supporting quote: "..." | strongest quote against: "..."

Counterfactual check. Name your second-best label. State what would have to appear in this post for
that label to be correct instead, then say whether the post contains it.

Then end with exactly one line, in this form:
Final label: Depression
or Final label: Neutral
or Final label: Happy"""

def build_v6_prompt(structure_text, post_text):
    return (
        V6_TEMPLATE
        .replace("{policy}", CLASSIFICATION_POLICY)
        .replace("{structure}", str(structure_text).strip())
        .replace("{text}", str(post_text))
    )

def stage_totals(stages):
    return {
        "input_tokens_total": sum(item["input_tokens"] for item in stages),
        "generated_tokens_total": sum(item["generated_tokens"] for item in stages),
        "generation_seconds_total": sum(item["elapsed_seconds"] for item in stages),
        "any_stage_hit_token_limit": any(item["hit_token_limit"] for item in stages),
    }

def run_v6_max_prompt(tokenizer, model, row, task_structure=None):
    result = chat_generate(
        tokenizer, model,
        [{"role": "system", "content": ANNOTATOR_SYSTEM_PROMPT},
         {"role": "user", "content": build_v6_prompt(
             task_structure["reasoning_structure"], row["phase2_original_text"]
         )}],
        max_new_tokens=V6_MAX_NEW_TOKENS,
        seed=stable_seed(row["example_id"], "v6_max_prompt", "single"),
        do_sample=False,
    )
    label = parse_final_label(result["text"])
    return {
        "sd_selected_modules": task_structure["selected_modules"],
        "sd_adapted_modules": task_structure["adapted_modules"],
        "sd_reasoning_structure": task_structure["reasoning_structure"],
        "sd_applied_reasoning": result["text"],
        "sd_final_answer": result["text"],
        # No Phase 1 label is ever shown, so the single verdict is both provisional and final.
        "provisional_label": label,
        "final_label": label,
        "vote_labels": "",
        "vote_unanimous": np.nan,
        **stage_totals([result]),
    }


## 3. 받은 최종 공통 계획
아래 문자열은 제공된 JSON 원본이다. 변경하거나 재생성하지 않는다.

In [ ]:
CACHED_PLAN_JSON = '{\n  "model_name": "NousResearch/Meta-Llama-3-8B-Instruct",\n  "module_bank": "emotion_domain_v1_18_modules",\n  "implement_prompt": "compact_v1_max_3_steps",\n  "task_description": "\\nTask: research-oriented, non-clinical emotion annotation of English social-media posts.\\n\\nEach instance is one post, consisting of a title and a body, written by a single author. An annotator\\nmust assign exactly one label from Depression, Neutral, Happy, according to the following policy.\\n\\nClassification Policy:\\n- Depression: unresolved sadness, hopelessness, emotional distress, emotional exhaustion, withdrawal, self-devaluation, or a clearly negative overall trajectory is dominant.\\n- Neutral: the text is mainly factual, routine, balanced, informational, or emotionally mild, without a dominant positive or distress-related state.\\n- Happy: happiness, relief, gratitude, accomplishment, fulfillment, or a clearly positive resolution is dominant.\\n\\nAssess the dominant emotional meaning of the full text. For mixed or shifting emotions, consider the overall trajectory and final takeaway. Do not decide from isolated words, and do not make clinical diagnoses or treatment recommendations.\\n\\nThe posts are real, user-written text. They vary greatly in length. A single post may cover several\\ntopics, quote or paraphrase other people, ask a question, give advice, report factual information,\\ndescribe a past episode that has since resolved, or mix positive and negative cues within the same\\nparagraph. The annotator must decide the dominant emotional meaning of the post as a whole, and must\\nbe able to point to the text that supports the decision.\\n\\nNo clinical diagnosis, medical inference, or treatment advice is produced. The output is a research\\nannotation label only.\\n",\n  "selected_modules": "Based on the task description, the following modules are crucial for solving the task:\\n\\n1. **Emotion Identification**: Separate the writer\'s own current emotional state from emotions they attribute to other people, to quoted text, or to hypothetical situations.\\n2. **Emotional Trajectory**: Locate the emotional stance the post opens with and the emotional stance it closes with, and state explicitly whether they differ.\\n3. **Emotional Takeaway**: Determine the final emotional takeaway the writer leaves the reader with, instead of averaging every emotional word in the post.\\n4. **Contextual Analysis**: Distinguish a past state the writer describes as changed or resolved from a state that is still ongoing at the time of writing.\\n5. **Purpose Identification**: Decide whether the post is primarily informational, procedural, advice-giving, or a question about a topic, rather than an expression of the writer\'s own current feelings.\\n6. **Sentiment Inversion**: Check for irony, sarcasm, dark humor, or understatement that inverts the literal sentiment of a phrase.\\n7. **Emotional Intensity**: Weigh explicit emotional statements against implicit signals such as sleep, appetite, isolation, disrupted routine, or loss of interest.\\n8. **Resolution Identification**: Identify markers of resolution: relief, gratitude, recovery, accomplishment, reconnection, or a plan the writer feels good about.\\n9. **Distress Identification**: Identify markers of unresolved distress: hopelessness, exhaustion, self-devaluation, numbness, withdrawal, or an appeal for help that the post never resolves.\\n10. **Label Ranking**: Rank all three candidate labels for this post and name which label is second-best, rather than stopping at the first label that seems to fit.\\n\\nThese modules are essential for accurately annotating the emotional meaning of the social media posts, considering the writer\'s emotional state, the emotional trajectory of the post, and the context in which the emotions are expressed.",\n  "adapted_modules": "Here are the adapted module descriptions:\\n\\n1. **Emotion Extraction**: Identify the writer\'s explicit and implicit emotional expressions, distinguishing between their own emotions and those attributed to others, quoted text, or hypothetical situations. Consider the context in which emotions are expressed to disentangle mixed or shifting emotions.\\n\\n2. **Emotional Arc**: Analyze the emotional trajectory of the post, identifying the emotional stance at the beginning and end of the text. Determine if the emotional tone shifts or remains consistent throughout, and note any significant changes or resolutions.\\n\\n3. **Emotional Essence**: Determine the dominant emotional takeaway the writer intends to convey, rather than simply averaging the emotional tone of the post. Consider the writer\'s purpose and the emotional resonance of the text as a whole.\\n\\n4. **Contextual Clarification**: Disentangle past events or experiences from the writer\'s current emotional state. Identify whether the writer is describing a resolved or ongoing situation, and consider the implications for their emotional expression.\\n\\n5. **Purpose Identification**: Determine the primary purpose of the post, such as informational, advisory, or questioning. This helps to contextualize the emotional expression and identify potential emotional cues.\\n\\n6. **Sentiment Reversal**: Detect instances of irony, sarcasm, dark humor, or understatement that invert the literal sentiment of a phrase. This module helps to uncover potential emotional contradictions or complexities.\\n\\n7. **Emotional Weight**: Evaluate the intensity and significance of explicit emotional statements, considering implicit signals such as sleep, appetite, isolation, or disrupted routine. This module helps to gauge the emotional impact of the post.\\n\\n8. **Resolution Detection**: Identify markers of resolution, such as relief, gratitude, recovery, accomplishment, reconnection, or a plan the writer feels good about. This module helps to determine if the post conveys a sense of closure or finality.\\n\\n9. **Distress Detection**: Identify markers of unresolved distress, including hopelessness, exhaustion, self-devaluation, numbness, withdrawal, or an appeal for help that remains unaddressed. This module helps to detect potential emotional pain or struggle.\\n\\n10. **Label Prioritization**: Rank the three candidate labels (Depression, Neutral, Happy) in order of likelihood, considering the emotional essence, emotional arc, and contextual factors. Provide a second-best label to ensure that the annotation is robust and defensible.\\n\\nThese adapted module descriptions aim to better capture the nuances of the task, focusing on the writer\'s emotional expression, the emotional trajectory of the post, and the context in which emotions are expressed.",\n  "reasoning_structure": "Here is the operationalized reasoning plan in JSON format:\\n\\n```\\n{\\n  \\"steps\\": [\\n    {\\n      \\"name\\": \\"Emotion Extraction\\",\\n      \\"sentence\\": \\"Extract the writer\'s explicit and implicit emotional expressions, distinguishing between their own emotions and those attributed to others, quoted text, or hypothetical situations.\\"\\n    },\\n    {\\n      \\"name\\": \\"Emotional Arc and Essence\\",\\n      \\"sentence\\": \\"Analyze the emotional trajectory of the post, identifying the emotional stance at the beginning and end, and determine the dominant emotional takeaway the writer intends to convey.\\"\\n    },\\n    {\\n      \\"name\\": \\"Contextual Clarification and Purpose Identification\\",\\n      \\"sentence\\": \\"Disentangle past events or experiences from the writer\'s current emotional state, identify the primary purpose of the post, and consider the implications for their emotional expression.\\"\\n    }\\n  ],\\n  \\"final_step\\": {\\n    \\"name\\": \\"Label Prioritization\\",\\n    \\"sentence\\": \\"Rank the three candidate labels (Depression, Neutral, Happy) in order of likelihood, considering the emotional essence, emotional arc, and contextual factors, and provide a second-best label to ensure the annotation is robust and defensible.\\"\\n  }\\n}\\n```\\n\\nThis plan consists of three main steps that extract the writer\'s emotional expressions, analyze the emotional trajectory and essence, and clarify the context and purpose of the post. The final step prioritizes the candidate labels based on the gathered evidence.",\n  "hit_token_limit": false\n}'
CACHED_PLAN = json.loads(CACHED_PLAN_JSON)
PROTOCOL_EVIDENCE = {'direct_cot_notebook_sha256': 'da49d75f9f4f372e1bca281e6e914ebfd13c6c847f911a969dc991b7c16c31b3', 'sd_notebook_sha256': '016ffbe9afba148d9faaf6e05417f6922ebbeee6e74ec53ad39d6c737f1a0a8e', 'cached_plan_original_sha256': 'bca5cadcc99adc299fc5370c0edd2b3130a4fd25d1c7df298faf61c695703f8b', 'protocol_source_sha256': 'dc0a659d39f9b7db2e0aa1e7f9eab6012137f1e6e32d535b43c0ff2bdf03f41a', 'runtime_source_sha256': 'd3559047e2b9e540761272949a2af0a3fbff6806a0a0ec7fb1bd8c1c16ec19b5'}


In [ ]:
"""Handoff validation, resumable generation, and full-denominator evaluation.

The notebook supplies the unchanged protocol functions before this runtime.
"""

import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd

METHODS = {"direct": "Llama 3 Direct", "cot": "Llama 3 CoT", "v6c": "Llama 3 SELF-DISCOVER v6c"}
FINAL_WEIGHT_SHA256 = "0eeb8b16aef6a5df3045c2924d4701305f0ca08dc85d8a66efc3daff33b253d6"


def digest_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def json_safe(value):
    if isinstance(value, dict):
        return {k: json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value


def save_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(json_safe(payload), ensure_ascii=False, indent=2, allow_nan=False))
    temporary.replace(path)


def save_csv(path, frame):
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary, index=False)
    temporary.replace(path)


def read_phase1_handoff(folder):
    handoff = json.loads((folder / "phase2_handoff.json").read_text())
    path = folder / "final_phase1_reasoning_input.csv"
    if digest_file(path) != handoff["input_sha256"]:
        raise ValueError("Phase 1 input does not match its handoff hash.")
    manifest = json.loads((folder / "experiment_manifest.json").read_text())
    if handoff["phase1_manifest"] != manifest:
        raise ValueError("Handoff and Phase 1 manifest disagree.")
    if manifest["model_file_sha256"].get("model.safetensors") != FINAL_WEIGHT_SHA256:
        raise ValueError("This export was not produced with the received final seed42 weights.")
    if manifest["temperature"] != 1.4801950079829693 or manifest["threshold"] != 0.7:
        raise ValueError("Phase 1 temperature/threshold changed.")
    frame = pd.read_csv(path, keep_default_na=False, dtype={"example_id": str})
    required = {"example_id", "target_label", "phase1_label", "phase1_confidence", "phase1_routed", "phase2_original_text"}
    if not required.issubset(frame.columns):
        raise ValueError(f"Missing columns: {required - set(frame.columns)}")
    if len(frame) != 9000 or len(frame) != handoff["expected_total_rows"]:
        raise ValueError("Expected exactly 9000 posts.")
    if frame.example_id.duplicated().any() or frame.example_id.str.strip().eq("").any():
        raise ValueError("Duplicate/empty example IDs.")
    if not frame.phase1_label.isin(LABELS).all() or not frame.target_label.isin(LABELS).all():
        raise ValueError("Invalid labels.")
    if frame.target_label.value_counts().to_dict() != dict.fromkeys(LABELS, 3000):
        raise ValueError("Expected 3000 reference examples per class.")
    frame["phase1_routed"] = frame.phase1_routed.map(normalize_bool)
    confidences = pd.to_numeric(frame.phase1_confidence, errors="raise")
    if not confidences.between(0, 1).all() or not np.array_equal(frame.phase1_routed, confidences < 0.7):
        raise ValueError("Routing flags do not match calibrated confidence < 0.70.")
    if int(frame.phase1_routed.sum()) != handoff["expected_routed_rows"]:
        raise ValueError("Routed count does not match the handoff.")
    saved = json.loads((folder / "phase1_summary.json").read_text())
    correct = frame.phase1_label.eq(frame.target_label)
    routed_error_count = int((~correct & frame.phase1_routed).sum())
    if (saved["total_rows"] != len(frame) or saved["routed_rows"] != int(frame.phase1_routed.sum())
            or saved["errors"] != int((~correct).sum()) or saved["routed_errors"] != routed_error_count
            or not np.isclose(saved["phase1_accuracy"], correct.mean(), atol=1e-12, rtol=0)
            or not np.isclose(saved["phase1_macro_f1"], f1_score(frame.target_label, frame.phase1_label,
                                                              labels=LABELS, average="macro"), atol=1e-12, rtol=0)):
        raise ValueError("Saved Phase 1 summary does not match its predictions.")
    routed = frame.loc[frame.phase1_routed].copy()
    if routed.phase2_original_text.str.strip().eq("").any():
        raise ValueError("A routed post has no original text.")
    previous = routed.get("phase2_input_was_truncated", pd.Series(False, index=routed.index)).map(normalize_bool)
    limited = routed.phase2_original_text.map(limit_reasoning_text)
    routed["phase2_original_text"] = limited.map(lambda pair: pair[0])
    routed["phase2_input_was_truncated"] = previous | limited.map(lambda pair: pair[1])
    return frame, routed.reset_index(drop=True), handoff, saved


def record_path(output, method, example_id):
    key = hashlib.sha256(str(example_id).encode()).hexdigest()
    return output / "records" / method / f"{key}.json"


def lock_run(output, manifest):
    output.mkdir(parents=True, exist_ok=True)
    path = output / "experiment_manifest.json"
    if path.exists():
        if json.loads(path.read_text()) != manifest:
            raise ValueError("Run settings/input/runtime changed. Preserve this run and use a new output directory.")
    else:
        if list(output.glob("records/*/*.json")) or list(output.glob("stages/*/*/*.json")) or list(output.glob("*_results.csv")):
            raise ValueError("Results without a matching manifest; refusing to mix runs.")
        save_json(path, manifest)
    return hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()


def cached_generation(base_generate, cache_dir, fingerprint):
    counter = 0

    def generate(tokenizer, model, messages, **kwargs):
        nonlocal counter
        counter += 1
        path = cache_dir / f"stage_{counter:02d}.json"
        request = {"messages": messages, "settings": kwargs}
        request_hash = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
        if path.exists():
            saved = json.loads(path.read_text())
            if saved["run_fingerprint"] != fingerprint or saved["request_sha256"] != request_hash:
                raise ValueError("Cached generation does not match this prompt/run.")
            return saved["generation"]
        generation = base_generate(tokenizer, model, messages, **kwargs)
        save_json(path, {"run_fingerprint": fingerprint, "request_sha256": request_hash,
                         "request": request, "generation": generation})
        return generation

    return generate


def load_method_records(output, method, routed, fingerprint, require_complete=True):
    expected = set(routed.example_id)
    rows = []
    for path in sorted((output / "records" / method).glob("*.json")):
        record = json.loads(path.read_text())
        if record.get("run_fingerprint") != fingerprint or record.get("method") != method:
            raise ValueError("Stale or mixed result record.")
        if record["example_id"] not in expected or path != record_path(output, method, record["example_id"]):
            raise ValueError("Unexpected result ID or filename.")
        if record.get("final_label") not in LABELS + [None]:
            raise ValueError("Invalid parsed label in result record.")
        rows.append(record)
    frame = pd.DataFrame(rows)
    observed = set(frame.example_id) if len(frame) else set()
    if len(rows) != len(observed):
        raise ValueError("Duplicate result IDs.")
    if require_complete and observed != expected:
        raise ValueError(f"{method}: {len(observed)}/{len(expected)} complete; finish generation first.")
    if len(frame):
        frame = frame.set_index("example_id").reindex([x for x in routed.example_id if x in observed]).reset_index()
    return frame


def write_method_results(full, routed, results, output, method):
    prediction_col = f"llama3_{method}_final_label"
    merged, summary = summarize_method(full, results.rename(columns={"final_label": prediction_col}),
                                       prediction_col, METHODS[method])
    save_csv(output / f"llama3_{method}_results.csv", results)
    save_csv(output / f"llama3_{method}_end_to_end_predictions.csv", merged)
    save_json(output / f"llama3_{method}_summary.json", summary)
    save_json(output / f"llama3_{method}_classification_report.json",
              classification_report(merged.target_label, merged.final_label, labels=LABELS,
                                    output_dict=True, zero_division=0))
    matrix = confusion_matrix(merged.target_label, merged.final_label, labels=LABELS)
    save_csv(output / f"llama3_{method}_confusion_matrix.csv",
             pd.DataFrame(matrix, columns=LABELS).assign(target_label=LABELS))
    return summary


def run_all_methods(full, routed, handoff, output):
    global chat_generate
    if not torch.cuda.is_available():
        raise RuntimeError("Select a GPU runtime before running Llama 3.")
    if hashlib.sha256(CACHED_PLAN_JSON.encode()).hexdigest() != PROTOCOL_EVIDENCE["cached_plan_original_sha256"]:
        raise ValueError("The frozen v6c plan changed.")
    manifest = {
        "protocol": "existing9000_final_llama3_direct_cot_v6c_v1",
        "phase1_handoff": handoff, "source_protocols": PROTOCOL_EVIDENCE,
        "model_name": MODEL_NAME, "model_revision": MODEL_REVISION,
        "methods": list(METHODS), "do_sample": False, "base_seed": BASE_SEED,
        "direct_cot_max_new_tokens_per_call": LLAMA3_MAX_NEW_TOKENS,
        "v6c_max_new_tokens": V6_MAX_NEW_TOKENS, "quantization": "nf4_double_quant_fp16",
        "max_reasoning_characters": MAX_REASONING_CHARACTERS,
        "reasoning_head_characters": REASONING_HEAD_CHARACTERS, "reasoning_tail_characters": REASONING_TAIL_CHARACTERS,
        "routed_id_sha256": hashlib.sha256("\n".join(sorted(routed.example_id)).encode()).hexdigest(),
        "total_rows": len(full), "routed_rows": len(routed), "new_plan_discovery": False,
        "parser": "existing_final_label_regex_then_unique_label_else_phase1_fallback",
        "torch": torch.__version__, "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0),
        "packages": {name: importlib.metadata.version(name) for name in (
            "transformers", "accelerate", "bitsandbytes", "numpy", "pandas", "scikit-learn")},
    }
    fingerprint = lock_run(output, manifest)
    plan_path = output / "frozen_llama3_v6c_plan.json"
    if plan_path.exists() and digest_file(plan_path) != PROTOCOL_EVIDENCE["cached_plan_original_sha256"]:
        raise ValueError("Saved frozen plan differs from the received original.")
    if not plan_path.exists():
        plan_path.write_text(CACHED_PLAN_JSON)
    done_by_method = {}
    for method in METHODS:
        saved = load_method_records(output, method, routed, fingerprint, require_complete=False)
        done_by_method[method] = set(saved.example_id) if len(saved) else set()
        print(f"{METHODS[method]}: {len(saved)} completed, {len(routed) - len(saved)} pending")
    if all(len(done) == len(routed) for done in done_by_method.values()):
        print("All generations are saved. No model loaded; proceed to the comparison table.")
        return
    base_generate = chat_generate
    tokenizer, model = load_chat_model(MODEL_NAME)
    if getattr(model.config, "_commit_hash", MODEL_REVISION) not in (None, MODEL_REVISION):
        raise ValueError("Resolved model revision differs from the frozen revision.")
    save_json(output / "loaded_model_environment.json", {
        "revision": getattr(model.config, "_commit_hash", MODEL_REVISION),
        "context_limit": int(model.config.max_position_embeddings),
        "attention_implementation": getattr(model.config, "_attn_implementation", None),
    })
    runners = {"direct": run_llama3_direct, "cot": run_llama3_cot, "v6c": run_v6_max_prompt}
    try:
        for method, runner in runners.items():
            for _, row in tqdm(routed.iterrows(), total=len(routed), desc=METHODS[method]):
                if row.example_id in done_by_method[method]:
                    continue
                path = record_path(output, method, row.example_id)
                chat_generate = cached_generation(base_generate, output / "stages" / method / path.stem, fingerprint)
                # Evaluation labels never enter the runner or the generation prompts.
                inference_row = {key: row[key] for key in ("example_id", "phase1_label", "phase2_original_text")}
                if method == "v6c":
                    generated = runner(tokenizer, model, inference_row, task_structure=CACHED_PLAN)
                else:
                    generated = runner(tokenizer, model, inference_row)
                save_json(path, {**generated, "example_id": row.example_id, "method": method,
                                 "run_fingerprint": fingerprint, "phase1_label": row.phase1_label,
                                 "phase2_input_was_truncated": bool(row.phase2_input_was_truncated),
                                 "generation_calls": 5 if method == "cot" else 1})
            results = load_method_records(output, method, routed, fingerprint)
            summary = write_method_results(full, routed, results, output, method)
            print(METHODS[method], "complete:", summary)
    finally:
        chat_generate = base_generate
        del model, tokenizer
        gc.collect()
        torch.cuda.empty_cache()


def export_all_results(full, routed, output):
    manifest = json.loads((output / "experiment_manifest.json").read_text())
    current_hash = digest_file(PHASE1_DIR / "final_phase1_reasoning_input.csv")
    if current_hash != manifest["phase1_handoff"]["input_sha256"]:
        raise ValueError("Phase 1 input changed after generation.")
    fingerprint = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()
    summaries = []
    costs = []
    for method in METHODS:
        results = load_method_records(output, method, routed, fingerprint)
        summaries.append(write_method_results(full, routed, results, output, method))
        costs.append({"method": METHODS[method], "routed_rows": len(routed),
                      "generation_calls": int(results.generation_calls.sum()),
                      **{key: float(results[key].sum()) for key in (
                          "input_tokens_total", "generated_tokens_total", "generation_seconds_total")},
                      "new_plan_discovery_calls": 0})
    summary = pd.DataFrame(summaries)
    save_csv(output / "llama3_reasoning_method_summary.csv", summary)
    save_csv(output / "llama3_generation_cost_summary.csv", pd.DataFrame(costs))
    return summary


## 4. 1단계 결과 검증
9,000건과 handoff 해시, 모델 출처, 라우팅 규칙을 확인한다. 오류가 있으면 LLM을 불러오기 전에 멈춘다.

In [ ]:
phase1_df, routed_df, handoff, phase1_summary = read_phase1_handoff(PHASE1_DIR)
display(pd.DataFrame([phase1_summary]))
print('세 방식 각각 평가할 사례:', len(routed_df))


## 5. 세 방식 실행 또는 이어서 실행
모델은 한 번만 불러온다. 이 셀은 수 시간이 걸릴 수 있다. 완료된 방법은 다시 실행하지 않는다.

In [ ]:
run_all_methods(phase1_df, routed_df, handoff, OUTPUT_DIR)


## 6. 최종 비교표
세 방식이 모두 완료됐는지 확인한 뒤 전체 9,000건 기준 결과표를 만든다. 원시 생성 기록도 같은 출력 폴더에 보존된다.

In [ ]:
summary = export_all_results(phase1_df, routed_df, OUTPUT_DIR)
display(summary)
print('Saved:', OUTPUT_DIR)


## 7. 저장 결과 검증 (CPU 전용)

이미 생성이 끝났으면 **아래 셀 하나만 실행**한다. 위 셀이나 모두 실행을 누를 필요 없다. GPU·모델 다운로드·재생성 없이 저장된 결과만 읽는다. Drive 연결 후 검증, 대응 통계 계산, 관련 CSV/JSON ZIP 다운로드를 진행한다.


In [ ]:
"""CPU-only audit of saved results. No model loading or generation."""

import hashlib
import json
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import binomtest
from sklearn.metrics import f1_score


def audit_9000_saved(root):
    labels = ["Depression", "Neutral", "Happy"]
    base = root / "additional_9000_20260930"
    p1dir = base / "phase1"
    p2dir = base / "phase2/llama3_direct_cot_v6c_v1"
    output = base / "audit_20260930"
    output.mkdir(parents=True, exist_ok=True)

    def sha(path):
        return hashlib.sha256(path.read_bytes()).hexdigest()

    def read(path):
        d = pd.read_csv(path, keep_default_na=False, dtype={"example_id": str})
        if d.example_id.duplicated().any() or d.example_id.eq("").any():
            raise ValueError(f"Duplicate/empty IDs: {path.name}")
        return d.set_index("example_id").sort_index()

    def truth(value):
        if str(value).lower() in ("true", "1"):
            return True
        if str(value).lower() in ("false", "0"):
            return False
        raise ValueError("Invalid Boolean")

    def parse(text):
        m = re.search(r"Final\s+label\s*\*{0,2}\s*:\s*\*{0,2}\s*(Depression|Neutral|Happy)\b", str(text), re.I)
        if m:
            return m[1].capitalize()
        unique = {x.capitalize() for x in re.findall(r"\b(Depression|Neutral|Happy)\b", str(text), re.I)}
        return next(iter(unique)) if len(unique) == 1 else ""

    def holm(values):
        order = np.argsort(values)
        adjusted = np.empty(len(values))
        adjusted[order] = np.minimum(1, np.maximum.accumulate(np.array(values)[order] * np.arange(len(values), 0, -1)))
        return adjusted

    source = p1dir / "final_phase1_reasoning_input.csv"
    handoff = json.loads((p1dir / "phase2_handoff.json").read_text())
    manifest = json.loads((p2dir / "experiment_manifest.json").read_text())
    assert sha(source) == handoff["input_sha256"]
    assert manifest["phase1_handoff"] == handoff
    phase1_manifest = json.loads((p1dir / "experiment_manifest.json").read_text())
    assert handoff["phase1_manifest"] == phase1_manifest
    assert phase1_manifest["model_file_sha256"]["model.safetensors"] == "0eeb8b16aef6a5df3045c2924d4701305f0ca08dc85d8a66efc3daff33b253d6"
    assert phase1_manifest["temperature"] == 1.4801950079829693
    assert phase1_manifest["threshold"] == 0.7
    assert sha(p2dir / "frozen_llama3_v6c_plan.json") == manifest["source_protocols"]["cached_plan_original_sha256"]
    fingerprint = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()
    assert manifest["model_revision"] == "53346005fb0ef11d3b6a83b12c895cca40156b6c"
    full = read(source)
    assert len(full) == 9000 == handoff["expected_total_rows"]
    assert full.target_label.value_counts().to_dict() == dict.fromkeys(labels, 3000)
    routed = full.phase1_routed.map(truth)
    assert routed.sum() == 111 == handoff["expected_routed_rows"]
    assert np.array_equal(routed, full.phase1_confidence < 0.7)
    vectors = {"Phase 1": full.phase1_label.eq(full.target_label).to_numpy()}
    summaries = []
    methods = {"direct": "direct_answer", "cot": "cot_final_answer", "v6c": "sd_final_answer"}
    saved_summary = pd.read_csv(p2dir / "llama3_reasoning_method_summary.csv").set_index("method")
    names = {"direct": "Llama 3 Direct", "cot": "Llama 3 CoT", "v6c": "Llama 3 SELF-DISCOVER v6c"}
    inventory = []
    for method, answer in methods.items():
        raw_path = p2dir / f"llama3_{method}_results.csv"
        final_path = p2dir / f"llama3_{method}_end_to_end_predictions.csv"
        raw, final = read(raw_path), read(final_path)
        assert set(raw.index) == set(full.index[routed])
        assert final.index.equals(full.index)
        assert raw.final_label.isin(labels + [""]).all()
        # Generation records omit reference labels; Phase 1 is the label source.
        # If a separate export includes labels, they must still match by ID.
        if "target_label" in raw.columns:
            assert raw["target_label"].equals(full.loc[raw.index, "target_label"])
        assert raw.phase1_label.equals(full.loc[raw.index, "phase1_label"])
        assert raw[answer].map(parse).equals(raw.final_label)
        recomputed = full.phase1_label.copy()
        recomputed.loc[raw.index] = raw.final_label.where(raw.final_label.isin(labels), full.loc[raw.index, "phase1_label"])
        assert recomputed.equals(final.final_label)
        assert final.target_label.equals(full.target_label)
        assert final.phase1_label.equals(full.phase1_label)
        expected_fallback = pd.Series(False, index=full.index)
        expected_fallback.loc[raw.index] = raw.final_label.eq("")
        assert final.phase2_fallback_to_phase1.map(truth).equals(expected_fallback)
        for eid, row in raw.iterrows():
            key = hashlib.sha256(eid.encode()).hexdigest()
            record = json.loads((p2dir / "records" / method / f"{key}.json").read_text())
            assert record["example_id"] == eid and record["run_fingerprint"] == fingerprint
            assert (record["final_label"] or "") == row.final_label
            stages = sorted((p2dir / "stages" / method / key).glob("stage_*.json"))
            assert len(stages) == (5 if method == "cot" else 1)
            generated_tokens, input_tokens = 0, 0
            for path in stages:
                stage = json.loads(path.read_text())
                assert stage["run_fingerprint"] == fingerprint
                assert hashlib.sha256(json.dumps(stage["request"], sort_keys=True).encode()).hexdigest() == stage["request_sha256"]
                settings = stage["request"]["settings"]
                assert settings["do_sample"] is False
                assert settings["max_new_tokens"] == (1536 if method == "v6c" else 1024)
                generated_tokens += stage["generation"]["generated_tokens"]
                input_tokens += stage["generation"]["input_tokens"]
            assert generated_tokens == row.generated_tokens_total and input_tokens == row.input_tokens_total
        current = recomputed.eq(full.target_label).to_numpy()
        vectors[method] = current
        stats = {"method": names[method], "total_rows": len(full), "routed_rows": len(raw),
                 "parse_failures": int(raw.final_label.eq("").sum()),
                 "end_to_end_accuracy": float(current.mean()),
                 "end_to_end_macro_f1": f1_score(full.target_label, recomputed, labels=labels, average="macro"),
                 "corrected": int((~vectors["Phase 1"] & current).sum()),
                 "introduced": int((vectors["Phase 1"] & ~current).sum())}
        for name, value in stats.items():
            if name != "method":
                assert np.isclose(saved_summary.loc[names[method], name], value, atol=1e-12, rtol=0), name
        summaries.append(stats)
        inventory.append({"method": method, "raw_sha256": sha(raw_path), "end_to_end_sha256": sha(final_path)})
    rows = []
    for a_name, b_name, family in [("Phase 1", m, "phase1_3") for m in methods] + [(m, "v6c", "sd_2") for m in ("direct", "cot")]:
        a, b = vectors[a_name], vectors[b_name]
        c, i = int((~a & b).sum()), int((a & ~b).sum())
        seed = int.from_bytes(hashlib.sha256((a_name+":"+b_name).encode()).digest()[:4], "big")
        draws = np.random.default_rng(seed).multinomial(len(full), [c/len(full), i/len(full), 1-(c+i)/len(full)], size=50000)
        low, high = np.percentile((draws[:, 0]-draws[:, 1])/len(full)*100, [2.5,97.5])
        rows.append({"reference": a_name, "method": b_name, "family": family, "corrected": c, "introduced": i,
                     "change_pp": (c-i)/len(full)*100, "ci_low_pp": low, "ci_high_pp": high,
                     "exact_p": binomtest(c, c+i, .5).pvalue if c+i else 1.0})
    table = pd.DataFrame(rows)
    for _, indices in table.groupby("family").groups.items():
        table.loc[indices, "holm_p"] = holm(table.loc[indices, "exact_p"].tolist())
    pd.DataFrame(summaries).to_csv(output / "row_verified_summary.csv", index=False)
    table.to_csv(output / "paired_statistics.csv", index=False)
    evidence = {"row_audit_passed": True, "total_rows": len(full), "routed_rows": int(routed.sum()),
                "generation_calls_checked": 777, "source_files": inventory, "statistics": table.to_dict("records")}
    (output / "audit_evidence.json").write_text(json.dumps(evidence, indent=2))

    # Include only the missing final results and final 9000 evidence, never weights or secrets.
    missing_final_dirs = [root / "phase2/llama2_cot_matched_1024_v1",
                          root / "mixed_emotion/phase2/llama2_cot_matched_1024_v1",
                          root / "mixed_emotion/phase2/llama2_direct_final_sd_v1",
                          root / "mixed_emotion/phase2/llama3_reasoning_comparison"]
    archive_path = output / "final_results_audit_package.zip"
    with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as archive:
        for directory in [p1dir, p2dir, *missing_final_dirs]:
            if not directory.exists():
                print("Missing saved directory:", directory)
                continue
            for path in sorted(directory.rglob("*")):
                if path.is_file() and path.suffix in (".csv", ".json") and "batches" not in path.parts:
                    archive.write(path, str(path.relative_to(root)))
        archive.write(output / "audit_evidence.json", "audit_evidence.json")
        archive.write(output / "paired_statistics.csv", "paired_statistics.csv")
    print("AUDIT PASSED: 9000 IDs, 111 routed posts, all three outputs and 777 generation calls verified.")
    print(table.to_string(index=False))
    print("AUDIT_EVIDENCE_JSON=" + json.dumps(evidence))
    print("Saved verification package:", archive_path)
    return archive_path


if __name__ == "__main__":
    from google.colab import drive, files
    drive.mount("/content/drive")
    audit_package = audit_9000_saved(Path("/content/drive/MyDrive/confidence_guided_llm_reasoning/outputs_final/unified_final"))
    files.download(str(audit_package))
